In [1]:
from dataloaders import SequentialCIFAR10
from torch.optim import AdamW
from tqdm.auto import tqdm
from torch import optim
from torch import nn
import torch
import torch.nn.functional as F
from losses import SupConLoss

/Users/alexanderbodner/Documents/Udesa/5to/vision avanzada/tp1_continual_learning/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seq_cifar = SequentialCIFAR10(data_root="./data", batch_size=32, buffer_size=200)
train_loader, val_loader = seq_cifar.get_train_val_loaders(task_id=0, use_buffer=False)


In [3]:
class CNN(nn.Module):
   def __init__(self, in_channels, embedding_dim):

       super(CNN, self).__init__()

       # 1st convolutional layer
       self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=3, padding=1)
       # Max pooling layer
       self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
       # 2nd convolutional layer
       self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
       # Fully connected layer
       self.fc1 = nn.Linear(16 * 8 * 8, embedding_dim)

   def forward(self, x):
       x = F.relu(self.conv1(x))
       x = self.pool(x)
       x = F.relu(self.conv2(x))
       x = self.pool(x)
       x = x.reshape(x.shape[0], -1)
       x = self.fc1(x)
       return x

In [4]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = CNN(in_channels=3, embedding_dim=32).to(device)
criterion = SupConLoss(tau=0.07)
optimizer = AdamW(model.parameters(), lr=5e-5)
model2 = CNN(in_channels=3, embedding_dim=32).to(device)


In [ ]:
def train(model, train_dataloader, val_dataloader, optimizer, criterion, lr_scheduler=None, num_epochs=10, device="cpu"):
    train_losses = []
    val_losses = []

    for epoch in tqdm(range(num_epochs), desc="Training"):
        model.train()
        running_loss = 0.0

        for images, labels in train_dataloader: 
            images = images.to(device)
            labels = labels.to(device)

            embeddings = model(images)
            loss = criterion(embeddings, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if lr_scheduler is not None:
                lr_scheduler.step()

            running_loss += loss.item()
        train_losses.append(running_loss / len(train_dataloader))

        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_dataloader:
                images = images.to(device)
                labels = labels.to(device)
                embeddings = model(images)
                loss = criterion(embeddings, labels)
                running_val_loss += loss.item()

        val_losses.append(running_val_loss / len(val_dataloader))

    return train_losses, val_losses

In [ ]:
train_losses, val_losses = train(model, train_loader, val_loader, optimizer, criterion, lr_scheduler=None, num_epochs=10, device=device)

Training:   0%|          | 0/10 [00:00<?, ?it/s]/Users/alexanderbodner/Documents/Udesa/5to/vision avanzada/tp1_continual_learning/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


entrena
entrena
entrena
entrena
entrena
entrena
entrena
entrena
entrena


Training:   0%|          | 0/10 [00:16<?, ?it/s]


KeyboardInterrupt: 